In [ ]:
#!/usr/bin/env python3.11
# -*- coding: utf-8 -*-
# Mahé C., Bretonnière H., Slezak E.
# 19-12-2025
# Observatoire de la Côte d'Azur, Laboratoire Lagrange, UMR 7293, Nice, France


In [ ]:
## Importation of modules

import pickle
import numpy as np
import matplotlib.pyplot as plt
import random
import time

import argparse
import yaml

import tensorflow_probability as tfp

from architecture.flow_architecture import model_flow
from architecture.vae_architecture import decoder

from utils.save_flow_class import FlowModel
from utils.score_gen_utils import score_gen_for_sample, score_normalised, score_power_spectrum
from utils.plot_utils import score_gen_multiple

tfd = tfp.distributions
tfb = tfp.bijectors


In [ ]:
### Hyperparameters ###
nb_made = 8
latent_dim = 32
nb_sample = 10000 ## Number of images to generate
nb_plot = 50 # Number of images to display
save = True

In [ ]:
# For the flow
run = 55

# For the decoder
type_loss = 'Fourier'
num_run = 220

In [ ]:
# Paths

path = './pickle_files/' ## Path for list of encodings and scores

flow_path_weights = f'./checkpoints/flow_model_{run}_end'
decoder_path_weigths = f'./checkpoints/{type_loss}_decoder_{num_run}_end'

In [ ]:
# Load of encodings and reconstruction scores 

with open(path + f'encod_list.pkl', 'rb') as f :
    dico_load = pickle.load(f)

    encod_np = dico_load['encod_np'][0] 

with open(path + f'scores_list.pkl', 'rb') as f :
    dico_load = pickle.load(f)

    scores_np = dico_load['scores_np'][0]

In [ ]:
### Model ###

maf = model_flow(nb_made, latent_dim)
flow_model = FlowModel(maf)
flow_model.load_weights(f'./checkpoints/flow_model_{run}_end')

samples = maf.sample(args.nb_sample)


In [ ]:
### Score VAE ###
rdm_encod = np.random.choice(np.arange(encod_np.shape[0]), 100, replace=False)
vae_samples = encod_np[rdm_encod]
            
encod_MSE, density_encod = score_gen_for_sample(maf, vae_samples, encod_np, scores_np, args.latent_dim, radius=0.5)    


In [ ]:
### Generations of new images ###

Decoder = decoder(latent_dim, generation_mode = True)
Decoder.load_weights(f'./checkpoints/{type_loss}_decoder_{num_run}_end')
decoded_img = Decoder(samples)


if save == True :
    dico_image = {'generated_images': decoded_img}

    with open(f'/workspace/cmahe/pickle_files/generated_images.pkl', 'wb') as f :
        pickle.dump(dico_image, f)


In [ ]:
### Visualisation ###

plot_samples = maf.sample(nb_plot)

score_sample = score_normalised(maf, plot_samples, encod_np, encod_MSE, density_encod, scores_np, latent_dim, radius=0.5)[0]

In [ ]:
nb_to_plot = int(np.sqrt(nb_plot))

fig, ax = plt.subplots (nb_to_plot, nb_to_plot, figsize = (30, 30))

nb_image = 0

for i in range(nb_to_plot) :
    for j in range(nb_to_plot) :

        ax[i,j].imshow(decoded_img[nb_image, :, :, 0])
        ax[i,j].set_title(rf'$S_g$ = {score_sample[nb_image]}', fontsize = 25)

        nb_image = nb_image + 1

if save == True :
    plt.savefig('test/generation_with_scores.png')

plt.close(fig)